<a href="https://colab.research.google.com/github/AnimeshPatidar/salesforce-o2c-sales-operations-analytics/blob/main/crm_clean_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [37]:
#importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [38]:
df = pd.read_csv('/content/sample_data/crm_project_dataset.csv')

In [39]:
df.head()

,customer_id,order_id,customer_name,company_name,region,product_name,discount_percent,order_status,sla_status,payment_status,...,shipment_date,sales_rep,unit_price,quantity,approval_status,quote_status,contract_type,discount_reason,order_amount,priority_level
0,1001,ORD-10001,Egon MacMearty,Roomm,West,Firewall,12,Pending,Met,Pending,...,NaN,Luella Enderby,100,85,Approved,Accepted,Enterprise,Bulk Order,8500,High
1,1002,ORD-10002,Debera Di Franceshci,Jabbersphere,East,Router,2,Cancelled,NaN,Pending,...,NaN,Ardath Sefton,200,38,Rejected,Expired,Enterprise,Bulk Order,7600,Medium
2,1003,ORD-10003,Audrye Cranefield,Thoughtbeat,East,VPN License,7,Completed,Met,Paid,...,2025-06-04,Edie MacPhee,200,39,Approved,Accepted,Monthly,Retention,7800,High
3,1004,ORD-10004,Avram Drakers,Feedfish,West,Firewall,13,Completed,Met,Paid,...,2026-03-11,Ryann Morillas,100,92,Approved,Accepted,Annual,Bulk Order,9200,Low
4,1005,ORD-10005,Niki Saunders,Flashspan,West,VPN License,13,Completed,Breached,Pending,...,2026-03-11,Niven Wannes,200,100,Approved,Accepted,Annual,Discount,20000,Low


In [40]:
#know about dataset
df.shape
df.info()
df.describe()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   customer_id       1000 non-null   int64 
 1   order_id          1000 non-null   object
 2   customer_name     1000 non-null   object
 3   company_name      1000 non-null   object
 4   region            1000 non-null   object
 5   product_name      1000 non-null   object
 6   discount_percent  1000 non-null   int64 
 7   order_status      1000 non-null   object
 8   sla_status        862 non-null    object
 9   payment_status    1000 non-null   object
 10  order_date        1000 non-null   object
 11  shipment_date     394 non-null    object
 12  sales_rep         1000 non-null   object
 13  unit_price        1000 non-null   int64 
 14  quantity          1000 non-null   int64 
 15  approval_status   1000 non-null   object
 16  quote_status      1000 non-null   object
 17  contract_type  

,0
customer_id,0
order_id,0
customer_name,0
company_name,0
region,0
product_name,0
discount_percent,0
order_status,0
sla_status,138
payment_status,0


In [41]:
#look out for missing values and fill it accordingly
df[df['sla_status'].isnull()]['order_status'].value_counts()

,count
order_status,
Cancelled,138


In [42]:
df['sla_status'] = df['sla_status'].fillna('n/a')
df.isnull().sum()


,0
customer_id,0
order_id,0
customer_name,0
company_name,0
region,0
product_name,0
discount_percent,0
order_status,0
sla_status,0
payment_status,0


In [43]:
# correct order and shipment dates
df['shipment_date'] = pd.to_datetime(df['shipment_date'])
df['order_date'] = pd.to_datetime(df['order_date'])
df[df['shipment_date'].isnull()]['order_status'].value_counts()

,count
order_status,
Pending,478
Cancelled,128


In [44]:
# Correct Dates logics by rows
shipped_df = df[df['shipment_date'].notnull()]
date_errors = shipped_df[shipped_df['order_date'] > shipped_df['shipment_date']]
print(f"Date logic error (Actaul shipped happened only): {len(date_errors)}")

Date logic error (Actaul shipped happened only): 50


In [45]:
df['days_to_ship'] = (df['shipment_date'] - df['order_date']).dt.days
print(df['days_to_ship'].mean())

11.416243654822335


In [46]:
#Question - How much total money did this region make? On an average, how fast is this region?

region_pf = df.groupby('region').agg({
    'order_amount': 'sum',
    'days_to_ship': 'mean'
}).sort_values(by='order_amount', ascending=False)

In [47]:
print(region_pf)

        order_amount  days_to_ship
region                            
East         2997650     13.250000
South        2534890     11.308411
West         2435430     10.439252
North        2345030     10.923913


In [48]:
# We need to see which regions are failing our customers( We need to know where money is at risk).
# By using a crosstab to get a count of met vs breached orders in percentage.
region_sla_report = pd.crosstab(df['region'], df['sla_status'], normalize='index') * 100
print("SLA  report in percentage by region")
print(region_sla_report)

SLA  report in percentage by region
sla_status   Breached        Met        n/a
region                                     
East        42.412451  45.136187  12.451362
North       43.495935  44.308943  12.195122
South       40.000000  44.400000  15.600000
West        40.485830  44.534413  14.979757


In [49]:
# We know risk right, now check if specific contract types are more likely to have payment delays.
payment_status = pd.crosstab(df['contract_type'], df['payment_status'], normalize='index') * 100
print("Payment Status % by Contract Type")
print(payment_status)

Payment Status % by Contract Type
payment_status    Overdue       Paid    Pending
contract_type                                  
Annual          15.677966  12.923729  71.398305
Enterprise      16.571429  11.428571  72.000000
Monthly         14.164306  16.430595  69.405099


In [50]:
# To indentify high priority accounts that have breached SLA and represent churn risk and need immediate attention.
risk = (df['priority_level'] == 'High') & (df['sla_status'] == 'Breached')
high_risk = df[risk]
impacted_revenue = high_risk['order_amount'].sum()
impacted_accounts = high_risk['company_name'].nunique()
print(f"Revenue at risk")
print(f"Total Revenue at risk ${round(impacted_revenue, 2)}")
print(f"Total high value accounts affected {impacted_accounts}")

Revenue at risk
Total Revenue at risk $932950
Total high value accounts affected 77


In [51]:

# CPQ for analyzing if discounts are tied to higher volume or just reducing margin.
cpq_discount_audit = df.groupby('discount_reason').agg({
    'discount_percent': 'mean',
    'order_amount': 'mean',
    'order_id': 'count'
}).rename(columns={'order_id': 'Deal_Count'})

# Sorting by highest discount to see the "cheapest" deals
print("CPQ audit based on Discount Reason")
print(cpq_audit.sort_values(by='discount_percent', ascending=False))

CPQ audit based on Discount Reason
                 discount_percent  order_amount  Deal_Count
discount_reason                                            
Retention               10.535032  10252.292994         157
Discount                10.466292  11616.292135         178
New Customer            10.025000  10561.000000         160
Seasonal Offer          10.018293   8528.963415         164
Bulk Order               9.755952  10649.583333         168
Deal Partner             9.225434  10162.138728         173


In [52]:
# Check which products are most discounted (to check discount to death situation)
product_discounts = df.groupby('product_name')['discount_percent'].mean().sort_values(ascending=False)
print("\nAverage Discount by Product (Pricing Strategy Check):")
print(product_discounts.head(5))


Average Discount by Product (Pricing Strategy Check):
product_name
VPN License       11.254777
Security Suite    10.344444
Router             9.848485
Switch             9.594118
Firewall           9.511905
Name: discount_percent, dtype: float64


In [53]:
# PREPARING FOR SALESFORCE IMPORT
# Convert to datetime again (just in case they reverted to strings)
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
df['shipment_date'] = pd.to_datetime(df['shipment_date'], errors='coerce')
df['order_date'] = df['order_date'].dt.strftime('%Y-%m-%d')
df['shipment_date'] = df['shipment_date'].dt.strftime('%Y-%m-%d')

# Clean up nulls for Salesforce then export (Salesforce prefers empty strings over NaN)
df_salesforce_ready = df.fillna('')
df_salesforce_ready.to_csv('Salesforce_Import_Final.csv', index=False)

from google.colab import files
files.download('Salesforce_Import_Final.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>